# Step 1.2a — Enriched Feature Matrix
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility  {Extension of Saadaoui (2026)}**

Builds the high-dimensional control matrix for the DoubleML and causal forest steps (Steps 2–5).

---

## Project structure
```
project/
├── data/
│   ├── Saadaoui_2026_JCE.dta
│   └── 02_features/
│       └── raw/          ← placing all manual downloads here
│           ├── bdi.csv         (Investing.com: ; delimited, mm/dd/yyyy, comma decimal)
│           └── gscpi.csv       (NY Fed: ; delimited, French dates, comma decimal)
└── notebooks/
    └── 02_feature_matrix.ipynb   ← this file
```

## Data sources

| Variable | Source | Format |
|---|---|---|
| Core dataset (PRI, WTI, controls) | Saadaoui (2026) `.dta` | 48 pre-built variables |
| VIX | Yahoo Finance `^VIX` | Daily → monthly mean |
| Gold (USD/oz) | Datahub.io LBMA | Monthly, auto-download |
| Brent crude | FRED `DCOILBRENTEU` | Daily → monthly mean |
| T-bill 3m, 10y Treasury, TED, REER BIS, INDPRO | FRED direct CSV | Monthly, auto-download |
| CNY/USD exchange rate | FRED `DEXCHUS` | Daily → monthly |
| US credit spread (BAA−10Y) | FRED `BAA10Y` | Monthly, auto-download |
| EM sovereign spread | FRED `BAMLEMCBPIOAS` | Monthly, auto-download |
| Baltic Dry Index | Investing.com manual CSV | `;` delimited, `mm/dd/yyyy`, comma decimal |
| GSCPI | NY Fed manual CSV | `;` delimited, French dates, comma decimal |
| Global EPU | Baker-Bloom-Davis Excel | Auto-download |

## Variables computed in this notebook

**From core dataset (Stata do-file equivalents):**
- `dllgop`  = `llgop.diff()`  — equivalent to Stata `d.llgop`
- `dl2lgop` = `l2lgop.diff()` — equivalent to Stata `d.l2lgop`

**Derived from new macro series:**
- `term_spread`      = `gs10 − tb3ms`
- `lreer`            = log(REER)
- `dvix`             = Δ VIX (monthly change)
- `lgold`            = log(gold price)
- `lbdi`             = log(BDI)
- `lip`              = log(INDPRO)
- `lepu`             = log(EPU)
- `dcny_usd`         = Δ log(CNY/USD) — first-differenced to ensure stationarity
- `cny_vol`          = 3-month rolling std of Δ log(CNY/USD)
- `brent_wti_spread` = log(Brent) − lwti — market segmentation proxy (avoids collinearity with outcome)

---
## Setup

In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pyreadstat
from pathlib import Path

# ── Robust path resolution ─────────────────────────────────────────────────────
# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
    project_root = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    project_root = cwd

RAW  = str((project_root / 'data' / '02_features' / 'raw').resolve())
PROC = str((project_root / 'data' / '02_features').resolve())
DTA  = str((project_root / 'data' / 'Saadaoui_2026_JCE.dta').resolve())

os.makedirs(RAW,  exist_ok=True)
os.makedirs(PROC, exist_ok=True)

# Sample window (buffer before 1990-01 allows lag construction)
START = '1989-01-01'
END   = '2022-03-01'

print('Setup complete.')
print(f'PROJECT_ROOT → {project_root}')
print(f'RAW          → {RAW}')
print(f'PROC         → {PROC}')

Setup complete.
PROJECT_ROOT → C:\Users\HP\Desktop\replication+contribution
RAW          → C:\Users\HP\Desktop\replication+contribution\data\02_features\raw
PROC         → C:\Users\HP\Desktop\replication+contribution\data\02_features


---
## Section A — Load & Download Raw Data
Each cell skips the download if its output file already exists. Safe to re-run from scratch.

### A1. Saadaoui Core Dataset
**Source:** Saadaoui (2026), *Journal of Comparative Economics*  
**File:** `data/Saadaoui_2026_JCE.dta`

**Controls computed here (not in .dta — generated on the fly in Stata with `d.llgop`, `d.l2lgop`):**
- `dllgop`  = `llgop.diff()`  → Stata `d.llgop`
- `dl2lgop` = `l2lgop.diff()` → Stata `d.l2lgop`

In [2]:
if not os.path.exists(DTA):
    raise FileNotFoundError(f'Core dataset not found: {DTA}')

df_core, _ = pyreadstat.read_dta(DTA)

# Convert Stata %tm (months since Jan 1960) to datetime
base = pd.Timestamp('1960-01-01')
df_core['date'] = df_core['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
df_core = df_core.set_index('date').sort_index()
df_core.index = df_core.index.to_period('M').to_timestamp()

# ── Compute Stata first-difference controls (not stored in .dta) ───────────────
# Stata: d.llgop  = llgop - L.llgop
# Stata: d.l2lgop = l2lgop - L.l2lgop
df_core['dllgop']  = df_core['llgop'].diff()
df_core['dl2lgop'] = df_core['l2lgop'].diff()

print(f'Core dataset: {df_core.shape[0]} obs, {df_core.shape[1]} variables')
print(f'Range: {df_core.index[0].date()} to {df_core.index[-1].date()}')
print()
print('controls_baseline variables:')
baseline = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
print(df_core[baseline].describe().round(4))

Core dataset: 386 obs, 50 variables
Range: 1990-01-01 to 2022-02-01

controls_baseline variables:
          llwip    dllgop    l2lwip   dl2lgop
count  386.0000  385.0000  386.0000  385.0000
mean     4.5735    0.0007    4.5714    0.0007
std      0.2519    0.0120    0.2521    0.0120
min      4.1334   -0.1457    4.1317   -0.1457
25%      4.3750   -0.0041    4.3743   -0.0042
50%      4.6146    0.0012    4.6137    0.0012
75%      4.7972    0.0066    4.7954    0.0066
max      4.9587    0.0438    4.9452    0.0438


### A2. VIX — CBOE Volatility Index
**Source:** Yahoo Finance `^VIX` via `yfinance`  
**Why:** Captures global financial stress and risk-appetite, the key transmission channel from geopolitical shocks to commodity markets (Bloom 2009).  
**Format:** yfinance ≥0.2 saves multi-level headers; loader handles both old and new formats.

In [20]:
P_VIX = f'{RAW}/vix.csv'

if not os.path.exists(P_VIX):
    vix = yf.download('^VIX', start=START, end=END, progress=False)
    if len(vix) > 50:
        if isinstance(vix.columns, pd.MultiIndex):
            vix.columns = [c[0] for c in vix.columns]
        vix[['Close']].rename(columns={'Close': 'vix'}).to_csv(P_VIX)
        print(f'Downloaded VIX: {len(vix)} daily rows')
    else:
        print('VIX download failed')
else:
    print('VIX: file exists, skipping.')

Downloaded VIX: 8103 daily rows


### A3. Gold Price (USD/oz)
**Source:** Datahub.io — LBMA monthly fixings  
**Why:** Safe-haven asset; gold prices rise during geopolitical uncertainty and capture an alternative transmission channel (Baur & Lucey 2010).  
**Format:** Standard CSV, `Date` + `Price` columns, already monthly.

In [21]:
P_GOLD = f'{RAW}/gold_monthly.csv'

if not os.path.exists(P_GOLD):
    url = 'https://datahub.io/core/gold-prices/_r/-/data/monthly-processed.csv'
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            with open(P_GOLD, 'w') as fh:
                fh.write(r.text)
            print('Downloaded Gold: monthly LBMA data')
        else:
            print(f'Gold download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'Gold download error: {e}')
else:
    print('Gold: file exists, skipping.')

Downloaded Gold: monthly LBMA data


### A4. Brent Crude Price
**Source:** FRED `DCOILBRENTEU` (Europe Brent Spot Price FOB, USD/barrel)  
**Coverage:** 1987-05 to present — full sample coverage  
**Why Brent-WTI spread and not raw Brent:**  
WTI (`lwti`) is the outcome. Including raw Brent as a control introduces near-perfect collinearity (ρ ≈ 0.99). The **Brent-WTI spread** (log difference) is orthogonal to WTI levels and captures geopolitical supply disruptions that widen the US domestic vs. global price gap (Kilian & Murphy 2014).  
**Format:** FRED direct CSV — clean, no header issues.

In [22]:
P_BRENT = f'{RAW}/brent_fred.csv'

if not os.path.exists(P_BRENT):
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU'
    try:
        brent_df = pd.read_csv(url, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
        brent_df = brent_df[~brent_df.index.duplicated(keep='first')]
        brent_df['DCOILBRENTEU'] = pd.to_numeric(brent_df['DCOILBRENTEU'], errors='coerce')
        brent_df = brent_df.dropna(subset=['DCOILBRENTEU'])
        brent_df[['DCOILBRENTEU']].to_csv(P_BRENT)
        print(f'Downloaded Brent (FRED): {len(brent_df)} daily rows')
        print(f'Coverage: {brent_df.index[0].date()} to {brent_df.index[-1].date()}')
    except Exception as e:
        print(f'FRED Brent download failed: {e}')
else:
    print('Brent (FRED): file exists, skipping.')

Downloaded Brent (FRED): 9874 daily rows
Coverage: 1987-05-20 to 2026-04-20


### A5. FRED Series — Core Macro & Financial Controls
**Source:** FRED direct CSV — no API key required  
**URL pattern:** `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<ID>`

| FRED ID | Variable | Why it belongs |
|---|---|---|
| `TB3MS` | 3-month T-bill yield (%) | US monetary policy stance; affects oil demand via investment costs |
| `GS10` | 10-year Treasury yield (%) | Long-run real rate channel; drives commodity futures pricing |
| `TEDRATE` | TED spread (Libor − T-bill, %) | Interbank credit risk; financial stress proxy (discontinued 2023 but covers full sample) |
| `RBUSBIS` | US REER, BIS broad index | Global dollar conditions; oil is dollar-priced (Kilian 2009) |
| `INDPRO` | US Industrial Production Index | Demand-side oil driver (Kilian 2009); standard in oil VAR literature |
| `DEXCHUS` | CNY/USD exchange rate | Direct FX channel of US-China geopolitical shocks |
| `BAA10Y` | Moody's Baa − 10Y spread (%) | US credit risk premium; geopolitical stress raises corporate borrowing costs |
| `BAMLEMCBPIOAS` | EM corporate OAS (bps) | Emerging-market financial conditions; China shocks transmit via EM channel |

In [23]:
FRED_SERIES = {
    'tb3ms'   : (f'{RAW}/tb3ms.csv',    'TB3MS'),
    'gs10'    : (f'{RAW}/gs10.csv',     'GS10'),
    'tedrate' : (f'{RAW}/tedrate.csv',  'TEDRATE'),
    'reer_bis': (f'{RAW}/reer_bis.csv', 'RBUSBIS'),
    'indpro'  : (f'{RAW}/indpro.csv',   'INDPRO'),
    'cny_usd' : (f'{RAW}/cny_usd.csv',  'DEXCHUS'),
    'baa10y'  : (f'{RAW}/baa10y.csv',   'BAA10Y'),
    'em_oas'  : (f'{RAW}/em_oas.csv',   'BAMLEMCBPIOAS'),
}

for key, (path, fred_id) in FRED_SERIES.items():
    if not os.path.exists(path):
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_id}'
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                with open(path, 'w') as fh:
                    fh.write(r.text)
                n = len(r.text.strip().split('\n')) - 1
                print(f'Downloaded {key:10s} ({fred_id}): {n} rows')
            else:
                print(f'{key:10s}: HTTP {r.status_code}')
        except Exception as e:
            print(f'{key:10s}: {e}')
    else:
        tmp = pd.read_csv(path, na_values=['.', 'NA', ''])
        print(f'{key:10s}: file exists ({len(tmp)} rows)')

tb3ms     : file exists (1107 rows)
gs10      : file exists (876 rows)
tedrate   : file exists (9407 rows)
reer_bis  : file exists (386 rows)
indpro    : file exists (1287 rows)
cny_usd   : file exists (11816 rows)
baa10y    : file exists (10514 rows)
em_oas    : file exists (794 rows)


### A6. Baltic Dry Index (BDI)
**Source:** Investing.com — manual download  
**Place file at:** `data/02_features/raw/bdi.csv`  
**URL:** https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f  
**Why:** BDI captures global shipping demand — a real-time proxy for world trade and commodity supply-chain conditions (Kilian 2009; Alizadeh & Muradoglu 2014). Geopolitical turning points in US-China trade directly affect shipping volumes.  
**Format:**
- Delimiter: `;`
- Date: `mm/dd/yyyy`
- Values: comma as **decimal** separator (e.g. `2568,3` = 2568.3 — NOT thousands separator)

In [24]:
P_BDI     = f'{RAW}/bdi.csv'
P_BDI_OUT = f'{RAW}/bdi_clean.csv'

if os.path.exists(P_BDI):
    # Read: semicolon delimited, comma is decimal separator
    bdi_raw = pd.read_csv(
        P_BDI,
        sep=';',
        decimal=',',
        header=0,
        na_values=['', ' ', '-']
    )
    # Normalise to exactly 2 columns regardless of what Investing.com puts in the header
    bdi_raw = bdi_raw.iloc[:, :2].copy()
    bdi_raw.columns = ['date', 'bdi']

    # Parse mm/dd/yyyy dates
    bdi_raw['date'] = pd.to_datetime(bdi_raw['date'], format='%m/%d/%Y', errors='coerce')
    bdi_raw['bdi']  = pd.to_numeric(bdi_raw['bdi'], errors='coerce')
    bdi_raw = bdi_raw.dropna(subset=['date', 'bdi']).set_index('date').sort_index()

    bdi_raw[['bdi']].to_csv(P_BDI_OUT)
    print(f'BDI: {len(bdi_raw)} rows | {bdi_raw.index[0].date()} to {bdi_raw.index[-1].date()}')
    print(f'     Value range: {bdi_raw["bdi"].min():.0f} – {bdi_raw["bdi"].max():.0f}')
else:
    print(f'BDI file not found — download manually.')
    print(f'URL: https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f')
    print(f'Save as: {P_BDI}')

BDI: 9479 rows | 1986-12-31 to 2025-05-02
     Value range: 1895 – 33154


### A7. Global Supply Chain Pressure Index (GSCPI)
**Source:** Federal Reserve Bank of New York  
**Place file at:** `data/02_features/raw/gscpi.csv`  
**URL:** https://www.newyorkfed.org/research/policy/gscpi  
**Why:** Measures global supply chain disruptions directly. Geopolitical turning points (trade wars, sanctions) cause supply chain stress — this is the key *transmission variable* (Benigno et al. 2022).  
**Coverage:** 1998-01 to present  
**Format:**
- Delimiter: `;`
- Dates: French abbreviations, mixed 2/4-digit years (e.g. `31-janv-98`, `28-Feb-1998`)
- Values: comma as **decimal** separator (e.g. `-1,09` = −1.09)

In [25]:
P_GSCPI     = f'{RAW}/gscpi.csv'
P_GSCPI_OUT = f'{RAW}/gscpi_cleaned.csv'
gscpi_series = None

if os.path.exists(P_GSCPI):
    # Read without skiprows — keep all data rows
    gscpi_df = pd.read_csv(P_GSCPI, sep=';', header=0, usecols=[0, 1],
                            na_values=['', ' ', '-'])
    date_col  = gscpi_df.columns[0]
    value_col = gscpi_df.columns[1]

    # French → English month abbreviations
    FR_TO_EN = {
        'janv': 'Jan', 'fév': 'Feb', 'févr': 'Feb',
        'mars': 'Mar', 'avr': 'Apr', 'mai': 'May',
        'juin': 'Jun', 'juil': 'Jul', 'août': 'Aug', 'aout': 'Aug',
        'sept': 'Sep', 'oct': 'Oct', 'nov': 'Nov',
        'déc': 'Dec', 'dec': 'Dec',
    }

    def parse_gscpi_date(s):
        s = str(s).strip().lower()
        for fr, en in FR_TO_EN.items():
            s = s.replace(fr, en)
        s = s.title()
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
        # pandas maps 2-digit years 69–99 → 1969–1999, 00–68 → 2000–2068
        # GSCPI starts 1998, so any year > 2030 means a 2-digit year was mis-mapped
        if pd.notna(dt) and dt.year > 2030:
            dt = dt.replace(year=dt.year - 100)
        return dt

    gscpi_df[date_col]  = gscpi_df[date_col].apply(parse_gscpi_date)
    gscpi_df[value_col] = (gscpi_df[value_col].astype(str)
                            .str.replace(',', '.', regex=False)
                            .pipe(pd.to_numeric, errors='coerce'))

    gscpi_df = (gscpi_df
                .dropna(subset=[date_col, value_col])
                .set_index(date_col)
                .sort_index())

    gscpi_series = gscpi_df[value_col].resample('MS').last().rename('gscpi')
    gscpi_series.to_csv(P_GSCPI_OUT)

    print(f'GSCPI: {gscpi_series.notna().sum()} months '
          f'| {gscpi_series.first_valid_index().date()} → {gscpi_series.last_valid_index().date()}')
    print(f'       Original kept at {P_GSCPI}')
    print(f'       Cleaned saved to {P_GSCPI_OUT}')
else:
    print(f'GSCPI not found — will be excluded from feature matrix.')
    print(f'Download from: https://www.newyorkfed.org/research/policy/gscpi')
    print(f'Save as: {P_GSCPI}')

GSCPI: 339 months | 1998-01-01 → 2026-03-01
       Original kept at C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/gscpi.csv
       Cleaned saved to C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/gscpi_cleaned.csv


### A8. Global Economic Policy Uncertainty Index (EPU)
**Source:** Baker, Bloom & Davis — policyuncertainty.com  
**URL:** `https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx`  
**Why:** Captures uncertainty about economic *policy* globally — distinct from financial volatility (VIX). Directly relevant to Saadaoui's research question: geopolitical turning points generate policy uncertainty that transmits to commodity markets (Baker, Bloom & Davis 2016).  
**Coverage:** Monthly from 1997-01

In [9]:
P_EPU = f'{RAW}/epu_global.xlsx'

if not os.path.exists(P_EPU):
    url = 'https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(P_EPU, 'wb') as fh:
                fh.write(r.content)
            print('Downloaded EPU global index')
        else:
            print(f'EPU download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'EPU download error: {e}')
else:
    print('EPU: file exists, skipping.')

EPU: file exists, skipping.


### A9. CDS Spreads — US and China 5-Year Sovereign
**Source:** Manual download from Bloomberg, Refinitiv, or Investing.com  
**Place files at:** `data/02_features/raw/us_cds_5y.csv` and `data/02_features/raw/china_cds_5y.csv`  
**Why:** Sovereign CDS directly price geopolitical risk. US-China tension widening their respective CDS spreads is a direct financial-market signal of geopolitical stress — exactly the transmission channel the thesis is studying.  
**Expected format:** Any delimiter, first column = date, second column = CDS value in basis points.

In [ ]:
# Sample window for these CDS files has limited coverage over our window frame
CDS_FILES = {
    'us_cds_5y'   : f'{RAW}/us_cds_5y.csv',
    'china_cds_5y': f'{RAW}/china_cds_5y.csv',
}

cds_series_store = {}

for name, path in CDS_FILES.items():
    if os.path.exists(path):
        try:
            cds_df = pd.read_csv(path)
            cds_df = cds_df.iloc[:, :2].copy()
            cds_df.columns = ['date', name]
            cds_df['date'] = pd.to_datetime(
                cds_df['date'].astype(str).str.replace(',', '.', regex=False),
                dayfirst=True, errors='coerce'
            )
            cds_df[name] = pd.to_numeric(
                cds_df[name].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )
            cds_df = cds_df.dropna(subset=['date']).set_index('date').sort_index()
            cds_m  = cds_df[name].resample('MS').last().rename(name)
            cds_series_store[name] = cds_m
            print(f'{name}: {cds_m.notna().sum()} months | {cds_m.first_valid_index().date()} → {cds_m.last_valid_index().date()}')
        except Exception as e:
            print(f'{name}: parse failed ({e})')
    else:
        print(f'{name}: not found (optional — add file to use)')

us_cds_5y: not found (optional — add file to use)
china_cds_5y: not found (optional — add file to use)


---
## Section B — Merge & Build Feature Matrix

### B1. Helper Functions

In [30]:
def ensure_datetimeindex(s):
    """Ensure a Series or DataFrame has a clean DatetimeIndex."""
    s = s.copy()
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, errors='coerce')
    s = s[s.index.notna()]
    s = s[~s.index.duplicated(keep='first')]
    return s.sort_index()

def to_monthly_mean(s):
    """Resample a daily Series to monthly mean."""
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').mean()

def to_monthly_last(s):
    """Resample a Series to month-start last value."""
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').last()

def load_fred(path, col_name):
    """Load a standard FRED CSV and return a monthly Series."""
    df = pd.read_csv(path, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    df = ensure_datetimeindex(df)
    s  = pd.to_numeric(df.iloc[:, 0], errors='coerce')
    return s.resample('MS').last().rename(col_name)

print('Helper functions defined.')

Helper functions defined.


### B2. Build All Monthly Series

In [31]:
monthly_series = {}

# ── VIX ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/vix.csv'):
    vix_d = pd.read_csv(f'{RAW}/vix.csv', index_col=0,
                         skiprows=lambda i: i in [1, 2], na_values=['.', 'NA', ''])
    vix_d = ensure_datetimeindex(vix_d)
    pcol  = [c for c in vix_d.columns if c.lower() in ('close', 'vix', 'adj close')][0]
    monthly_series['vix'] = to_monthly_mean(vix_d[pcol]).rename('vix')
    print(f"vix        : {monthly_series['vix'].notna().sum()} months")

# ── Gold ──────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/gold_monthly.csv'):
    gold_d = pd.read_csv(f'{RAW}/gold_monthly.csv', index_col=0, parse_dates=True)
    gold_d = ensure_datetimeindex(gold_d)
    gold_s = to_monthly_last(gold_d.iloc[:, 0])
    monthly_series['lgold'] = np.log(gold_s.replace(0, np.nan)).rename('lgold')
    print(f"lgold      : {monthly_series['lgold'].notna().sum()} months")

# ── Brent (FRED DCOILBRENTEU) — reads brent_fred.csv ─────────────────────────
# NOTE: We compute brent_wti_spread in B3, not raw lbrent, to avoid collinearity
if os.path.exists(f'{RAW}/brent_fred.csv'):
    brent_d = pd.read_csv(f'{RAW}/brent_fred.csv', index_col=0,
                           parse_dates=True, na_values=['.', 'NA', ''])
    brent_d = ensure_datetimeindex(brent_d)
    brent_d['DCOILBRENTEU'] = pd.to_numeric(brent_d['DCOILBRENTEU'], errors='coerce')
    brent_m = to_monthly_mean(brent_d['DCOILBRENTEU'])
    monthly_series['lbrent'] = np.log(brent_m.replace(0, np.nan)).rename('lbrent')
    print(f"lbrent     : {monthly_series['lbrent'].notna().sum()} months (full raw history)")
else:
    print("lbrent     : brent_fred.csv not found — run A4 first")

# ── FRED core series ──────────────────────────────────────────────────────────
fred_load = {
    'tb3ms'   : f'{RAW}/tb3ms.csv',
    'gs10'    : f'{RAW}/gs10.csv',
    'tedrate' : f'{RAW}/tedrate.csv',
    'reer'    : f'{RAW}/reer_bis.csv',
    'indpro'  : f'{RAW}/indpro.csv',
    'cny_usd' : f'{RAW}/cny_usd.csv',
    'us_spread': f'{RAW}/baa10y.csv',
    'em_oas'  : f'{RAW}/em_oas.csv',
}
for col, path in fred_load.items():
    if os.path.exists(path):
        monthly_series[col] = load_fred(path, col)
        print(f"{col:12s}: {monthly_series[col].notna().sum()} months")

# ── BDI ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/bdi_clean.csv'):
    bdi_d = pd.read_csv(f'{RAW}/bdi_clean.csv', index_col=0, parse_dates=True)
    bdi_d = ensure_datetimeindex(bdi_d)
    monthly_series['bdi'] = to_monthly_mean(bdi_d['bdi']).rename('bdi')
    print(f"bdi        : {monthly_series['bdi'].notna().sum()} months")

# ── GSCPI ─────────────────────────────────────────────────────────────────────
if gscpi_series is not None and gscpi_series.notna().any():
    monthly_series['gscpi'] = gscpi_series
    print(f"gscpi      : {monthly_series['gscpi'].notna().sum()} months")

# ── EPU ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/epu_global.xlsx'):
    try:
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', header=None)
        header_row = epu_df[epu_df[0] == 'Year'].index[0]
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', skiprows=header_row)
        epu_df.columns = ['Year', 'Month', 'GEPU_current', 'GEPU_ppp']
        epu_df['Year']  = pd.to_numeric(epu_df['Year'],  errors='coerce')
        epu_df['Month'] = pd.to_numeric(epu_df['Month'], errors='coerce')
        epu_df = epu_df.dropna(subset=['Year', 'Month'])
        epu_df['date'] = pd.to_datetime(
            epu_df[['Year', 'Month']].assign(Day=1)
            .rename(columns={'Year': 'year', 'Month': 'month', 'Day': 'day'})
        )
        # Use PPP-weighted index (more appropriate for a global measure)
        epu_col = 'GEPU_ppp' if 'GEPU_ppp' in epu_df.columns else 'GEPU_current'
        epu_s = epu_df.set_index('date')[epu_col].sort_index()
        epu_s = ensure_datetimeindex(epu_s)
        monthly_series['epu'] = epu_s.resample('MS').last().rename('epu')
        print(f"epu        : {monthly_series['epu'].notna().sum()} months")
    except Exception as e:
        print(f'EPU load error: {e}')

# ── Optional CDS spreads (if files were provided) ─────────────────────────────
for name, cds_s in cds_series_store.items():
    monthly_series[name] = cds_s
    print(f"{name:12s}: {cds_s.notna().sum()} months")

print(f'\nTotal series collected: {len(monthly_series)}')

vix        : 386 months
lgold      : 2319 months
lbrent     : 468 months (full raw history)
tb3ms       : 1107 months
gs10        : 876 months
tedrate     : 433 months
reer        : 386 months
indpro      : 1287 months
cny_usd     : 544 months
us_spread   : 484 months
em_oas      : 37 months
bdi        : 462 months
gscpi      : 339 months
epu        : 347 months

Total series collected: 14


### B3. Derived Variables

In [32]:
derived = []

# ── Yield curve slope ─────────────────────────────────────────────────────────
if 'gs10' in monthly_series and 'tb3ms' in monthly_series:
    monthly_series['term_spread'] = (monthly_series['gs10'] - monthly_series['tb3ms']).rename('term_spread')
    derived.append('term_spread')

# ── Log REER ──────────────────────────────────────────────────────────────────
if 'reer' in monthly_series:
    monthly_series['lreer'] = np.log(monthly_series['reer'].replace(0, np.nan)).rename('lreer')
    derived.append('lreer')

# ── VIX first difference (fear shock) ────────────────────────────────────────
if 'vix' in monthly_series:
    monthly_series['dvix'] = monthly_series['vix'].diff().rename('dvix')
    derived.append('dvix')

# ── Log BDI ───────────────────────────────────────────────────────────────────
if 'bdi' in monthly_series:
    monthly_series['lbdi'] = np.log(monthly_series['bdi'].replace(0, np.nan)).rename('lbdi')
    derived.append('lbdi')

# ── Log Industrial Production ─────────────────────────────────────────────────
if 'indpro' in monthly_series:
    monthly_series['lip'] = np.log(monthly_series['indpro'].replace(0, np.nan)).rename('lip')
    derived.append('lip')

# ── Log EPU ───────────────────────────────────────────────────────────────────
if 'epu' in monthly_series:
    monthly_series['lepu'] = np.log(monthly_series['epu'].replace(0, np.nan)).rename('lepu')
    derived.append('lepu')

# ── CNY/USD: first-difference of log (ensures stationarity) ──────────────────
# Level CNY/USD is I(1) — use Δlog(CNY/USD) to capture FX flow direction
if 'cny_usd' in monthly_series:
    cny_log = np.log(monthly_series['cny_usd'].replace(0, np.nan))
    monthly_series['dcny_usd'] = cny_log.diff().rename('dcny_usd')
    # 3-month rolling volatility of the log change — captures FX uncertainty
    monthly_series['cny_vol'] = cny_log.diff().rolling(3, min_periods=2).std().rename('cny_vol')
    derived.extend(['dcny_usd', 'cny_vol'])
    # Remove raw level — it is non-stationary and not useful as a control
    del monthly_series['cny_usd']

# ── Brent-WTI spread (log difference) ────────────────────────────────────────
# Used INSTEAD of raw lbrent — avoids near-perfect collinearity with lwti
# Spread > 0 means global crude is more expensive than US domestic crude
if 'lbrent' in monthly_series:
    lwti_s = ensure_datetimeindex(df_core['lwti'].copy())
    lwti_s.index = lwti_s.index.to_period('M').to_timestamp()
    lbrent_aligned = monthly_series['lbrent'].reindex(lwti_s.index)
    monthly_series['brent_wti_spread'] = (lbrent_aligned - lwti_s).rename('brent_wti_spread')
    # Remove raw lbrent — brent_wti_spread is what we use in regressions
    del monthly_series['lbrent']
    derived.append('brent_wti_spread')

print(f'Derived variables: {derived}')

Derived variables: ['term_spread', 'lreer', 'dvix', 'lbdi', 'lip', 'lepu', 'dcny_usd', 'cny_vol', 'brent_wti_spread']


### B4. Merge with Core Dataset

> **Look-ahead bias prevention:** All newly added macro controls are lagged 1 month (`shift(1)`) before merging. At time *t*, the model sees only controls known at *t−1*. This is critical for causal validity of the DoubleML step (see Plan Step 10).  
> **The Saadaoui baseline controls (`llwip`, `dllgop`, `l2lwip`, `dl2lgop`) are already pre-lagged in the `.dta` file — do NOT lag them again.**

In [33]:
# Align core index to month-start timestamps
df_core.index = pd.to_datetime(df_core.index).to_period('M').to_timestamp()
df_enriched = df_core.copy()

for col_name, series in monthly_series.items():
    s = series.copy()
    s = ensure_datetimeindex(s)
    s.index = s.index.to_period('M').to_timestamp()
    s.name = col_name
    df_enriched = df_enriched.join(s, how='left')

# Lag all new macro controls by 1 month
new_ctrl_cols = [c for c in monthly_series.keys() if c in df_enriched.columns]
df_enriched[new_ctrl_cols] = df_enriched[new_ctrl_cols].shift(1)

# Restrict to Saadaoui sample: 1990-01 to 2022-01
df_enriched = df_enriched.loc['1990-01-01':'2022-01-01']

print(f'Enriched dataset: {df_enriched.shape[0]} obs × {df_enriched.shape[1]} variables')
print(f'Range: {df_enriched.index[0].date()} to {df_enriched.index[-1].date()}')

Enriched dataset: 385 obs × 71 variables
Range: 1990-01-01 to 2022-01-01


### B5. Coverage Audit

In [34]:
new_vars = [c for c in monthly_series.keys() if c in df_enriched.columns]

audit = pd.DataFrame({
    'n_obs'     : df_enriched[new_vars].notna().sum(),
    'pct_filled': (df_enriched[new_vars].notna().mean() * 100).round(1),
    'first_obs' : df_enriched[new_vars].apply(lambda c: c.first_valid_index()),
    'last_obs'  : df_enriched[new_vars].apply(lambda c: c.last_valid_index()),
}).sort_values('pct_filled', ascending=False)

print(audit.to_string())

# Variables with 0% coverage must be dropped before ML steps
dead_vars = audit[audit['pct_filled'] == 0].index.tolist()
if dead_vars:
    print(f'\n⚠  ZERO-COVERAGE variables (will be DROPPED from controls_macro): {dead_vars}')

sparse_vars = audit[(audit['pct_filled'] > 0) & (audit['pct_filled'] < 80)].index.tolist()
if sparse_vars:
    print(f'\n⚠  SPARSE variables (<80% coverage — use only in robustness specs): {sparse_vars}')

                  n_obs  pct_filled  first_obs   last_obs
vix                 384        99.7 1990-02-01 2022-01-01
lgold               384        99.7 1990-02-01 2022-01-01
tb3ms               384        99.7 1990-02-01 2022-01-01
gs10                384        99.7 1990-02-01 2022-01-01
tedrate             384        99.7 1990-02-01 2022-01-01
indpro              384        99.7 1990-02-01 2022-01-01
us_spread           384        99.7 1990-02-01 2022-01-01
term_spread         384        99.7 1990-02-01 2022-01-01
bdi                 384        99.7 1990-02-01 2022-01-01
dcny_usd            384        99.7 1990-02-01 2022-01-01
cny_vol             384        99.7 1990-02-01 2022-01-01
lbdi                384        99.7 1990-02-01 2022-01-01
lip                 384        99.7 1990-02-01 2022-01-01
brent_wti_spread    384        99.7 1990-02-01 2022-01-01
dvix                383        99.5 1990-03-01 2022-01-01
lreer               336        87.3 1994-02-01 2022-01-01
reer          

### B6. Variable Role Dictionary

> Zero-coverage variables are automatically excluded from `controls_macro`.  
> Sparse variables (coverage <80%) are kept in `controls_macro` but flagged.

In [35]:
# Variables with zero coverage — exclude from all ML specs
zero_coverage = [c for c in new_vars
                 if df_enriched[c].notna().sum() == 0]

VAR_ROLES = {
    'outcome'           : ['lwti'],
    'treatment'         : ['lpri'],
    'instrument'        : ['d2pri', 'd2pri_jp'],

    # Exact Saadaoui (2026) specification from Stata do-file:
    # locproj lwti lpri llwip d.llgop l2lwip d.l2lgop
    # dllgop  = d.llgop  = llgop.diff()
    # dl2lgop = d.l2lgop = l2lgop.diff()
    'controls_baseline' : ['llwip', 'dllgop', 'l2lwip', 'dl2lgop'],

    # Alliance controls from Appendix C (dlpri_* from .dta)
    'controls_alliance' : [c for c in df_enriched.columns if c.startswith('dlpri_')],

    # New macro controls — zero-coverage excluded automatically
    'controls_macro'    : [
        c for c in [
            # Financial stress
            'vix', 'dvix',
            # Yield curve
            'gs10', 'tb3ms', 'term_spread',
            # Credit risk
            'tedrate', 'us_spread', 'em_oas',
            # Dollar / FX
            'lreer', 'dcny_usd', 'cny_vol',
            # Commodities
            'lgold', 'brent_wti_spread', 'lbdi',
            # Real economy
            'lip',
            # Supply chains & uncertainty
            'gscpi', 'lepu',
            # Optional CDS (included if file was provided)
            'us_cds_5y', 'china_cds_5y',
        ]
        if c in df_enriched.columns and c not in zero_coverage
    ],

    # Variables present but <80% coverage — robustness use only
    'controls_sparse'   : [
        c for c in new_vars
        if 0 < df_enriched[c].notna().mean() * 100 < 80
        and c not in zero_coverage
    ],
}

VAR_ROLES['controls_all_ml'] = (
    VAR_ROLES['controls_baseline']
    + VAR_ROLES['controls_alliance']
    + VAR_ROLES['controls_macro']
)

# Warn about zero-coverage
if zero_coverage:
    VAR_ROLES['EXCLUDED_zero_coverage'] = zero_coverage

print('Variable inventory:')
for role, cols in VAR_ROLES.items():
    print(f'  {role:28s} ({len(cols):2d}): {cols}')
print(f'\nTotal ML control variables: {len(VAR_ROLES["controls_all_ml"])}')

Variable inventory:
  outcome                      ( 1): ['lwti']
  treatment                    ( 1): ['lpri']
  instrument                   ( 2): ['d2pri', 'd2pri_jp']
  controls_baseline            ( 4): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
  controls_alliance            (11): ['dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk']
  controls_macro               (16): ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'us_spread', 'lreer', 'dcny_usd', 'cny_vol', 'lgold', 'brent_wti_spread', 'lbdi', 'lip', 'gscpi', 'lepu']
  controls_sparse              ( 3): ['gscpi', 'epu', 'lepu']
  controls_all_ml              (31): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk', 'vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'us_spread', 'lreer', 'dcny

### B7. Export

In [17]:
OUT_CSV   = f'{PROC}/feature_matrix.csv'
OUT_ROLES = f'{PROC}/var_roles.json'

df_enriched.to_csv(OUT_CSV)

with open(OUT_ROLES, 'w') as fh:
    json.dump({k: list(v) for k, v in VAR_ROLES.items()}, fh, indent=2)

print(f'Saved: {OUT_CSV}')
print(f'Saved: {OUT_ROLES}')
print(f'Feature matrix shape: {df_enriched.shape}')

Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/feature_matrix.csv
Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/var_roles.json
Feature matrix shape: (385, 71)


---
## Appendix — Descriptive Statistics & Correlations

In [18]:
macro_vars = [v for v in VAR_ROLES['controls_macro'] if v in df_enriched.columns]
if macro_vars:
    print('=== Descriptive statistics (macro controls) ===')
    display(df_enriched[macro_vars].describe().round(3))

=== Descriptive statistics (macro controls) ===


,vix,dvix,gs10,tb3ms,term_spread,tedrate,us_spread,lreer,dcny_usd,cny_vol,lgold,brent_wti_spread,lbdi,lip,gscpi,lepu
count,384.000,383.000,384.000,384.000,384.000,384.000,384.000,336.000,384.000,384.000,384.000,384.000,384.000,384.000,288.000,300.000
mean,19.490,-0.007,4.301,2.548,1.753,0.467,2.350,4.526,0.001,0.007,6.470,0.654,9.580,4.477,-0.052,4.747
std,7.665,4.158,2.036,2.232,1.090,0.356,0.729,0.082,0.023,0.023,0.672,0.278,0.575,0.161,0.944,0.456
min,10.125,-16.283,0.620,0.010,-0.530,0.060,1.300,4.362,-0.035,0.000,5.545,0.137,7.938,4.100,-1.530,3.948
25%,13.973,-1.731,2.512,0.160,0.878,0.230,1.790,4.454,-0.003,0.000,5.876,0.383,9.250,4.434,-0.620,4.364
50%,17.620,-0.276,4.210,2.085,1.695,0.380,2.215,4.540,-0.000,0.002,6.272,0.654,9.662,4.530,-0.280,4.732
75%,23.099,1.182,5.905,4.812,2.662,0.572,2.762,4.586,0.000,0.006,7.145,0.920,10.048,4.597,0.143,5.047
max,62.639,38.108,8.890,7.900,3.760,3.150,6.100,4.689,0.405,0.234,7.585,1.099,10.369,4.645,4.490,6.036


In [19]:
# Correlation with log WTI — exploratory only, not causal
if macro_vars:
    print('=== Correlations with log WTI (exploratory) ===')
    corr = (
        df_enriched[macro_vars + ['lwti']]
        .corr()['lwti']
        .drop('lwti')
        .sort_values(key=abs, ascending=False)
        .round(3)
    )
    print(corr.to_string())

=== Correlations with log WTI (exploratory) ===
lgold               0.652
brent_wti_spread    0.639
lip                 0.557
lreer              -0.555
lbdi                0.550
tb3ms              -0.499
gs10               -0.498
us_spread           0.250
dcny_usd           -0.125
vix                -0.122
cny_vol            -0.100
term_spread         0.091
lepu                0.082
tedrate            -0.075
gscpi               0.070
dvix               -0.033
